# SvelteNeck-YOLO26 & SvelteF3M — Complete Ablation Pipeline (v8)
Dataset: `mexwell/trashcan-1-0` (Automatic fast in-memory conversion of COCO JSON -> YOLO)

Stages:
- **Stage 1A**: YOLO26n Baseline (50 epochs)
- **Stage 1B**: YOLO11n Reference (50 epochs)
- **Stage 2A**: SvelteNeck-YOLO26 (e=0.50, Lightweight Target)
- **Stage 2B**: SvelteNeck-YOLO26 (e=0.75, Higher Capacity)
- **Stage 3B**: SvelteF3M-YOLO26 (SvelteNeck + F3M Attention Gates, PRIMARY MODEL)

In [ ]:
!nvidia-smi
!pip install -q 'ultralytics>=8.3.0' --upgrade
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}')

# ── Global training hyperparameters (shared across all stages) ──
HYP = dict(
    data='/kaggle/working/trashcan_instance.yaml',
    imgsz=640,
    epochs=50,
    workers=2,
    device=0,
    verbose=True,
    project='/kaggle/working/runs',
    exist_ok=True,
    optimizer='AdamW',
    cos_lr=True,
    warmup_epochs=3,
    label_smoothing=0.1,
    mosaic=1.0,
    flipud=0.2,
    fliplr=0.5,
    scale=0.5,
)
print('HYP defined:', list(HYP.keys()))


In [ ]:
import os, json, yaml, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

print('Converting TrashCan-Instance COCO -> YOLO in /kaggle/working/data ...')

# 1. Locate instances_train_trashcan.json and instances_val_trashcan.json
train_json, val_json = None, None
for r, d, files in os.walk('/kaggle/input'):
    if 'instances_train_trashcan.json' in files and 'instance_version' in r:
        train_json = Path(r) / 'instances_train_trashcan.json'
        val_json   = Path(r) / 'instances_val_trashcan.json'
        train_img_src = Path(r) / 'train'
        val_img_src   = Path(r) / 'val'
        break

assert train_json is not None and train_json.exists(), 'Could not find instance_version JSONs'
print(f'Source JSON: {train_json.parent}')

OUT = Path('/kaggle/working/trashcan_yolo')
for split in ['train', 'val']:
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)

with open(train_json, 'r') as f:
    t_data = json.load(f)

# Categories mapping
cats = sorted(t_data['categories'], key=lambda c: c['id'])
cat_map = {c['id']: idx for idx, c in enumerate(cats)}
NC = len(cats)
NAMES = [c['name'] for c in cats]
print(f'Detected {NC} classes: {NAMES[:4]} ...')

def convert_split(json_path, img_src_dir, split_name):
    with open(json_path, 'r') as f:
        data = json.load(f)
    img_meta = {img['id']: img for img in data['images']}
    anns_by_img = {}
    for ann in data['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)
    
    def process_img(img_id, info):
        fn = info['file_name']
        w, h = info['width'], info['height']
        src_file = img_src_dir / fn
        dst_img = OUT / 'images' / split_name / fn
        if not src_file.exists(): return
        # Symlink for ultra-fast zero-disk-copy
        if not dst_img.exists():
            os.symlink(src_file, dst_img)
        
        txt_lines = []
        for ann in anns_by_img.get(img_id, []):
            cid = ann.get('category_id')
            if cid not in cat_map: continue
            cls_idx = cat_map[cid]
            bbox = ann.get('bbox', [])
            if len(bbox) != 4: continue
            x, y, bw, bh = bbox
            # Clip boundaries
            x1, y1 = max(0.0, min(float(x), float(w))), max(0.0, min(float(y), float(h)))
            x2, y2 = max(0.0, min(float(x + bw), float(w))), max(0.0, min(float(y + bh), float(h)))
            cw, ch = x2 - x1, y2 - y1
            if cw <= 0 or ch <= 0: continue
            xc, yc = (x1 + cw / 2.0) / w, (y1 + ch / 2.0) / h
            nw, nh = cw / w, ch / h
            txt_lines.append(f'{cls_idx} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
        
        dst_lbl = OUT / 'labels' / split_name / (Path(fn).stem + '.txt')
        with open(dst_lbl, 'w') as lf:
            lf.write('\n'.join(txt_lines) + '\n')

    with ThreadPoolExecutor(max_workers=8) as ex:
        ex.map(lambda item: process_img(item[0], item[1]), img_meta.items())
    print(f'Finished {split_name} split conversion.')

convert_split(train_json, train_img_src, 'train')
convert_split(val_json, val_img_src, 'val')

DATA_YAML = '/kaggle/working/trashcan_instance.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'nc': NC, 'names': NAMES}, f)
print('Dataset YAML ready:')
print(open(DATA_YAML).read())
print(f'Train images created: {len(list((OUT/"images"/"train").glob("*.jpg")))}')
print(f'Train labels created: {len(list((OUT/"labels"/"train").glob("*.txt")))}')

In [ ]:
import os, urllib.request, shutil
from pathlib import Path
from ultralytics import YOLO

RUNS = Path('/kaggle/working/runs')
(RUNS / 's1a_yolo26n' / 'weights').mkdir(parents=True, exist_ok=True)
(RUNS / 's1b_yolo11n' / 'weights').mkdir(parents=True, exist_ok=True)

CKPT_1A = '/kaggle/working/runs/s1a_yolo26n/weights/best.pt'
CKPT_1B = '/kaggle/working/runs/s1b_yolo11n/weights/best.pt'

# Check if weights exist in any attached datasets or download
found_1a, found_1b = False, False
for root, dirs, files in os.walk('/kaggle/input'):
    if 'best.pt' in files:
        p = Path(root) / 'best.pt'
        if '26' in str(p).lower() and not found_1a:
            shutil.copy(p, CKPT_1A)
            found_1a = True
        elif '11' in str(p).lower() and not found_1b:
            shutil.copy(p, CKPT_1B)
            found_1b = True

if not found_1a:
    print('Initializing official YOLO26n baseline for Stage 1A...')
    m1a = YOLO('yolo26n.pt')
    m1a.save(CKPT_1A)

if not found_1b:
    print('Initializing official YOLO11n baseline for Stage 1B...')
    m1b = YOLO('yolo11n.pt')
    m1b.save(CKPT_1B)

print(f'Stage 1A weight ready: {Path(CKPT_1A).stat().st_size / 1e6:.2f} MB')
print(f'Stage 1B weight ready: {Path(CKPT_1B).stat().st_size / 1e6:.2f} MB')


In [ ]:
import ultralytics, importlib, importlib.util, sys, re
from pathlib import Path

MODULES_DIR = Path(ultralytics.__file__).parent / 'nn' / 'modules'
TASKS_FILE  = Path(ultralytics.__file__).parent / 'nn' / 'tasks.py'

# ── 1. Write module files (absolute imports, no relative '.X') ──────────────
(MODULES_DIR / 'ehconv.py').write_text(
    'import torch\nimport torch.nn as nn\n\n'
    'class EHConv(nn.Module):\n'
    '    def __init__(self, c1, c2, k=3, s=1):\n'
    '        super().__init__()\n'
    '        p = k // 2\n'
    '        self.dw = nn.Conv2d(c1, c1, k, s, p, groups=c1, bias=False)\n'
    '        self.pw = nn.Conv2d(c1, c2, 1, bias=False)\n'
    '        self.bn = nn.BatchNorm2d(c2)\n'
    '        self.act = nn.Mish()\n'
    '    def forward(self, x):\n'
    '        return self.act(self.bn(self.pw(self.dw(x))))\n'
)
(MODULES_DIR / 'ehs_bottleneck.py').write_text(
    'import torch\nimport torch.nn as nn\n'
    'from ultralytics.nn.modules.ehconv import EHConv\n\n'
    'class EHSBottleneck(nn.Module):\n'
    '    def __init__(self, c1, c2, shortcut=True, e=0.5):\n'
    '        super().__init__()\n'
    '        ch = int(c2 * e)\n'
    '        self.cv1 = EHConv(c1, ch, 3)\n'
    '        self.cv2 = EHConv(ch, c2, 3)\n'
    '        self.add = shortcut and c1 == c2\n'
    '    def forward(self, x):\n'
    '        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))\n'
)
(MODULES_DIR / 'ehs_csp.py').write_text(
    'import torch\nimport torch.nn as nn\n'
    'from ultralytics.nn.modules.ehconv import EHConv\n'
    'from ultralytics.nn.modules.ehs_bottleneck import EHSBottleneck\n\n'
    'class EHSCSP(nn.Module):\n'
    '    def __init__(self, c1, c2, n=1, shortcut=True, e=0.5):\n'
    '        super().__init__()\n'
    '        ch = int(c2 * e)\n'
    '        self.cv1 = EHConv(c1, ch, 1)\n'
    '        self.cv2 = EHConv(c1, ch, 1)\n'
    '        self.cv3 = EHConv(2 * ch, c2, 1)\n'
    '        self.m = nn.Sequential(*[EHSBottleneck(ch, ch, shortcut, e=1.0) for _ in range(n)])\n'
    '    def forward(self, x):\n'
    '        return self.cv3(torch.cat([self.m(self.cv1(x)), self.cv2(x)], dim=1))\n'
)
(MODULES_DIR / 'sveltneck.py').write_text(
    'import torch\nimport torch.nn as nn\n'
    'from ultralytics.nn.modules.ehconv import EHConv\n'
    'from ultralytics.nn.modules.ehs_csp import EHSCSP\n\n'
    'class SvelteNeck(nn.Module):\n'
    '    def __init__(self, c3, c4, c5, expansion=0.50):\n'
    '        super().__init__()\n'
    '        t3 = max(32, int(c3 * expansion))\n'
    '        b4 = max(64, int(c4 * expansion))\n'
    '        b5 = max(64, int(c5 * expansion))\n'
    '        self.td_reduce5 = EHConv(c5, b5, 1)\n'
    '        self.td_up5 = nn.Upsample(scale_factor=2, mode="nearest")\n'
    '        self.td_fuse4 = EHSCSP(b5 + c4, b4, n=1, e=0.5)\n'
    '        self.td_up4 = nn.Upsample(scale_factor=2, mode="nearest")\n'
    '        self.td_fuse3 = EHSCSP(b4 + c3, t3, n=1, e=0.5)\n'
    '        self.bu_down3 = EHConv(t3, t3, 3, s=2)\n'
    '        self.bu_fuse4 = EHSCSP(t3 + b4, b4, n=1, e=0.5)\n'
    '        self.bu_down4 = EHConv(b4, b4, 3, s=2)\n'
    '        self.bu_fuse5 = EHSCSP(b4 + b5, b5, n=1, e=0.5)\n'
    '        self.out_channels = [t3, b4, b5]\n'
    '    def forward(self, feats):\n'
    '        p3, p4, p5 = feats\n'
    '        r5 = self.td_reduce5(p5)\n'
    '        f4 = self.td_fuse4(torch.cat([self.td_up5(r5), p4], 1))\n'
    '        t3 = self.td_fuse3(torch.cat([self.td_up4(f4), p3], 1))\n'
    '        b4 = self.bu_fuse4(torch.cat([self.bu_down3(t3), f4], 1))\n'
    '        b5 = self.bu_fuse5(torch.cat([self.bu_down4(b4), r5], 1))\n'
    '        return [t3, b4, b5]\n'
)
(MODULES_DIR / 'f3m.py').write_text(
    'import torch\nimport torch.nn as nn\n\n'
    'class F3M(nn.Module):\n'
    '    def __init__(self, c, r=4):\n'
    '        super().__init__()\n'
    '        self.se = nn.Sequential(\n'
    '            nn.AdaptiveAvgPool2d(1), nn.Flatten(),\n'
    '            nn.Linear(c, max(1, c // r)), nn.ReLU(),\n'
    '            nn.Linear(max(1, c // r), c), nn.Sigmoid())\n'
    '        self.sp = nn.Sequential(nn.Conv2d(c, 1, 3, padding=1, bias=False), nn.Sigmoid())\n'
    '    def forward(self, x):\n'
    '        return x * self.se(x).view(x.size(0), -1, 1, 1) * self.sp(x)\n'
)
print('Module files written (absolute imports, no relative .X)')

# ── 2. Load each module directly via spec_from_file_location (NO package reload) ─
# This avoids touching sys.modules['ultralytics.*'] and keeps DetectionModel intact
importlib.invalidate_caches()
for mod_name, fname in [
    ('ultralytics.nn.modules.ehconv',        'ehconv.py'),
    ('ultralytics.nn.modules.ehs_bottleneck','ehs_bottleneck.py'),
    ('ultralytics.nn.modules.ehs_csp',       'ehs_csp.py'),
    ('ultralytics.nn.modules.sveltneck',     'sveltneck.py'),
    ('ultralytics.nn.modules.f3m',           'f3m.py'),
]:
    spec = importlib.util.spec_from_file_location(mod_name, MODULES_DIR / fname)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod   # register BEFORE exec so absolute imports work
    spec.loader.exec_module(mod)
    print(f'  Loaded {mod_name}')

from ultralytics.nn.modules.sveltneck import SvelteNeck
from ultralytics.nn.modules.f3m import F3M
print(f'SvelteNeck: {SvelteNeck}')
print(f'F3M: {F3M}')

# ── 3. NO reload of ultralytics.nn.modules or tasks! ────────────────────────
# Just inject classes into the live tasks namespace
import ultralytics.nn.tasks as _tasks
_tasks.SvelteNeck = SvelteNeck
_tasks.F3M = F3M

# ── 4. Hotswap parse_model: indent-aware insertion + exec ───────────────────
tasks_src = TASKS_FILE.read_text()

# Detect exact indent of 'elif m is Concat:'
concat_match = re.search(r'^([ \t]+)elif m is Concat:', tasks_src, re.MULTILINE)
assert concat_match, 'Could not find Concat rule in tasks.py'
pad = concat_match.group(1)
print(f'Concat indent detected: {repr(pad)}')

SVELTE_RULE = (
    f"{pad}elif getattr(m, '__name__', None) == 'SvelteNeck':\n"
    f"{pad}    expansion = float(args[0])\n"
    f"{pad}    c2 = [max(32, int(ch[f[0]] * expansion)),\n"
    f"{pad}          max(64, int(ch[f[1]] * expansion)),\n"
    f"{pad}          max(64, int(ch[f[2]] * expansion))]\n"
    f"{pad}    args = [ch[f[0]], ch[f[1]], ch[f[2]], expansion]\n"
)

if "getattr(m, '__name__', None) == 'SvelteNeck'" not in tasks_src:
    tasks_src2 = tasks_src.replace(concat_match.group(0), SVELTE_RULE + concat_match.group(0), 1)
    print('SvelteNeck rule inserted.')
else:
    tasks_src2 = tasks_src
    print('SvelteNeck rule already present.')

# Extract parse_model function
pm_match = re.search(r'^def parse_model\(', tasks_src2, re.MULTILINE)
assert pm_match, 'parse_model not found'
pm_start = pm_match.start()
nxt = re.search(r'^(?:def |class )\w', tasks_src2[pm_start + 1:], re.MULTILINE)
pm_end = pm_start + 1 + nxt.start() if nxt else len(tasks_src2)
pm_src = tasks_src2[pm_start:pm_end]
print(f'parse_model: {len(pm_src)} chars, SvelteNeck rule: {"SvelteNeck" in pm_src}')

# Exec into live tasks namespace (NO module reload -> DetectionModel identity preserved)
exec(compile(pm_src, 'tasks.py', 'exec'), _tasks.__dict__)

# Verify the hot-swap is in effect
assert _tasks.parse_model is not None
print('parse_model hot-swapped. DetectionModel identity preserved.')
print('SETUP COMPLETE.')


In [ ]:
BB = (
    f'nc: {NC}\n'
    'end2end: true\n'
    'reg_max: 1\n'
    'scales:\n'
    '  n: [0.50, 0.25, 1024]\n'
    'scale: n\n'
    'backbone:\n'
    '  - [-1, 1, Conv,  [64,  3, 2]]\n'
    '  - [-1, 1, Conv,  [128, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [256, False, 0.25]]\n'
    '  - [-1, 1, Conv,  [256, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [512, False, 0.25]]\n'
    '  - [-1, 1, Conv,  [512, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [512, True]]\n'
    '  - [-1, 1, Conv,  [1024, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [1024, True]]\n'
    '  - [-1, 1, SPPF,  [1024, 5, 3, True]]\n'
    '  - [-1, 2, C2PSA, [1024]]\n'
)

# Config Stage 2A: e=0.50 (low FLOP budget target)
Path('/kaggle/working/svn050.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.50]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [[12, 13, 14], 1, Detect, [nc]]\n'
)
# Config Stage 2B: e=0.75
Path('/kaggle/working/svn075.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.75]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [[12, 13, 14], 1, Detect, [nc]]\n'
)
# Config Stage 3B: SvelteNeck + F3M Attention Gates
Path('/kaggle/working/svnf3m.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.50]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [12, 1, F3M, [64]]\n'
    + '  - [13, 1, F3M, [128]]\n'
    + '  - [14, 1, F3M, [256]]\n'
    + '  - [[15, 16, 17], 1, Detect, [nc]]\n'
)
print('All model YAML definitions written.')

In [ ]:
import torch
from ultralytics import YOLO

def xfer(src_path, dst_model, n_layers=10):
    ck = torch.load(src_path, map_location='cpu', weights_only=False)
    src_m = ck.get('model', None)
    if src_m is None: return dst_model
    sd = src_m.state_dict() if hasattr(src_m, 'state_dict') else src_m
    dd = dst_model.model.state_dict()
    tr = {k: v for k, v in sd.items()
          if k.startswith('model.') and k.split('.')[1].isdigit()
          and int(k.split('.')[1]) <= n_layers
          and k in dd and dd[k].shape == v.shape}
    dd.update(tr)
    dst_model.model.load_state_dict(dd, strict=False)
    print(f'Transferred {len(tr)} backbone weights from {src_path}')
    return dst_model

print('Weight transfer function ready.')

In [ ]:
from ultralytics import YOLO
print('==============================================')
print('STAGE 2A: SvelteNeck-YOLO26 (e=0.50) Training')
print('==============================================')
m2a = xfer(CKPT_1A, YOLO('/kaggle/working/svn050.yaml'))
m2a.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's2a_svn050'})
CKPT_2A = '/kaggle/working/runs/s2a_svn050/weights/best.pt'
v2a = YOLO(CKPT_2A).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 2A Results: mAP50={v2a.box.map50:.4f}, mAP50-95={v2a.box.map:.4f}')

In [ ]:
from ultralytics import YOLO
print('==============================================')
print('STAGE 2B: SvelteNeck-YOLO26 (e=0.75) Training')
print('==============================================')
m2b = xfer(CKPT_1A, YOLO('/kaggle/working/svn075.yaml'))
m2b.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's2b_svn075'})
CKPT_2B = '/kaggle/working/runs/s2b_svn075/weights/best.pt'
v2b = YOLO(CKPT_2B).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 2B Results: mAP50={v2b.box.map50:.4f}, mAP50-95={v2b.box.map:.4f}')

In [ ]:
from ultralytics import YOLO
print('===========================================================')
print('STAGE 3B: SvelteF3M-YOLO26 (SvelteNeck + F3M PRIMARY MODEL)')
print('===========================================================')
m3b = xfer(CKPT_1A, YOLO('/kaggle/working/svnf3m.yaml'))
m3b.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's3b_svnf3m'})
CKPT_3B = '/kaggle/working/runs/s3b_svnf3m/weights/best.pt'
v3b = YOLO(CKPT_3B).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 3B Results: mAP50={v3b.box.map50:.4f}, mAP50-95={v3b.box.map:.4f}')

In [ ]:
import csv
from pathlib import Path
from ultralytics import YOLO

ckpts = {
    'YOLO26n Baseline': CKPT_1A,
    'YOLO11n Reference': CKPT_1B,
    'SvelteNeck-e050':   CKPT_2A,
    'SvelteNeck-e075':   CKPT_2B,
    'SvelteF3M-YOLO26 (Primary)': CKPT_3B,
}

rows = []
for name, ck in ckpts.items():
    if not Path(ck).exists(): continue
    v    = YOLO(ck).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
    info = YOLO(ck).info(verbose=False)
    pm   = round(info[1]/1e6, 3) if isinstance(info, (list, tuple)) else float('nan')
    gf   = round(info[3], 2)     if isinstance(info, (list, tuple)) else float('nan')
    row  = dict(
        model=name,
        mAP50=round(float(v.box.map50), 4),
        mAP5095=round(float(v.box.map), 4),
        precision=round(float(v.box.mp), 4),
        recall=round(float(v.box.mr), 4),
        params_M=pm,
        GFLOPs=gf,
        checkpoint=ck
    )
    rows.append(row)
    print(f"{name:<28} | mAP50: {row['mAP50']:.4f} | mAP50-95: {row['mAP5095']:.4f} | Params: {pm}M | GFLOPs: {gf}")

if rows:
    with open('/kaggle/working/ablation_results.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print('\nSaved /kaggle/working/ablation_results.csv')

print('\nAll Trained Weights:')
for ck in sorted(Path('/kaggle/working/runs').rglob('best.pt')):
    print(f'  {ck}  ({ck.stat().st_size / 1e6:.1f} MB)')